# U-Net Droplet / Nucleus Training Notebook v6.2

Clean version with data-driven NPC puncta clipping, flat-field droplet foreground correction, watershed diagnostics, filled nucleus labels, and memory-mapped TIFF loading.


In [ ]:
# ============================================================
# 1. Core imports
# ============================================================

from dataclasses import dataclass
from pathlib import Path
import os
import math
import time
import gc
import shutil
import inspect

import numpy as np
import pandas as pd

import tifffile as tiff
import matplotlib.pyplot as plt

from scipy import ndimage as ndi
from skimage import filters, morphology, measure
from skimage.filters import threshold_local
from skimage.measure import regionprops
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy import ndimage as ndi

import tensorflow as tf
from tensorflow.keras import layers, models

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices("GPU")))


In [ ]:
# ============================================================
# 2. Configuration
# ============================================================

@dataclass
class TrainingConfig:
    # ----------------------------
    # Project paths
    # ----------------------------
    data_root: Path = Path("/home/tdeibert/Data/Machine_Learning_Dev/")
    model_root: Path = data_root / "Models"
    image_root: Path = data_root / "Images"
    out_root: Path = data_root / "Outputs"

    image_filename: str = "control_extract_1.1.tif"

    # ----------------------------
    # Image metadata
    # Expected shape: (T, Z, C, Y, X)
    # ----------------------------
    pixel_size_um: float = 0.108
    n_channels: int = 3

    # Channel order from current FOV visualization
    membrane_channel_idx: int = 0
    nucleus_channel_idx: int = 1
    npc_channel_idx: int = 2

    # ----------------------------
    # Reproducibility
    # ----------------------------
    seed: int = 42

    # ----------------------------
    # Focus filtering
    # ----------------------------
    use_focus_filter: bool = True
    min_focus_score: float = 0.0005
    exclude_edge_z_planes: bool = True
    edge_z_exclusion: int = 2

    # ----------------------------
    # NPC puncta clipping
    # ----------------------------
    use_data_driven_tail: bool = True
    histogram_bins: int = 512
    histogram_smooth_window: int = 9
    tail_start_quantile: float = 0.80
    min_clip_quantile: float = 0.95
    max_clip_quantile: float = 0.995
    puncta_clip_quantile: float = 0.985  # fallback only

    # ----------------------------
    # Droplet segmentation
    # ----------------------------
    droplet_blur_sigma: float = 8.0
    droplet_illumination_sigma: float = 50.0
    droplet_foreground_threshold: float = 0.50
    min_droplet_area_um2: float = 500.0
    droplet_hole_area_um2: float = 8000.0
    droplet_closing_radius_um: float = 1.0
    watershed_min_distance_um: float = 3.0

    # ----------------------------
    # Nucleus segmentation
    # ----------------------------
    min_nucleus_area_um2: float = 120.0
    max_nucleus_area_um2: float = 1500.0
    nucleus_blur_sigma: float = 1.5
    nucleus_threshold_method: str = "otsu"
    nucleus_closing_radius_um: float = 1.5
    nucleus_hole_area_um2: float = 200.0

    # ----------------------------
    # Patch extraction
    # 512 px = 55.3 µm at 0.108 µm/px
    # ----------------------------
    patch_size: int = 512
    patch_jitter_px: int = 128
    patches_per_plane: int = 24
    nucleus_patch_fraction: float = 0.70
    droplet_patch_fraction: float = 0.20
    hard_negative_patch_fraction: float = 0.10
    min_label_fraction: float = 0.005

    # ----------------------------
    # Model / training
    # ----------------------------
    model_name: str = "unet_droplet_nucleus_v6_2"
    num_classes: int = 3
    batch_size: int = 2
    epochs: int = 50
    learning_rate: float = 1e-4
    validation_fraction: float = 0.20

    @property
    def image_file(self):
        return self.image_root / self.image_filename

    @property
    def training_root(self):
        return self.out_root / f"training_patches_{self.model_name}"

    @property
    def image_patch_dir(self):
        return self.training_root / "images"

    @property
    def label_patch_dir(self):
        return self.training_root / "labels"

    @property
    def qc_dir(self):
        return self.out_root / f"qc_{self.model_name}"

    @property
    def best_model_path(self):
        return self.model_root / f"{self.model_name}_best.keras"

    @property
    def final_model_path(self):
        return self.model_root / f"{self.model_name}_final.keras"


cfg = TrainingConfig()

for p in [
    cfg.data_root,
    cfg.model_root,
    cfg.image_root,
    cfg.out_root,
    cfg.training_root,
    cfg.image_patch_dir,
    cfg.label_patch_dir,
    cfg.qc_dir,
]:
    p.mkdir(parents=True, exist_ok=True)

# Backward-compatible aliases used by older snippets
DATA_ROOT = cfg.data_root
MODEL_ROOT = cfg.model_root
IMAGE_ROOT = cfg.image_root
OUT_ROOT = cfg.out_root
IMAGE_FILE = cfg.image_file
PIXEL_SIZE_UM = cfg.pixel_size_um
MEM_CH = cfg.membrane_channel_idx
NUC_CH = cfg.nucleus_channel_idx
NPC_CH = cfg.npc_channel_idx
PATCH_SIZE = cfg.patch_size
NUM_CHANNELS = cfg.n_channels
NUM_CLASSES = cfg.num_classes
BATCH_SIZE = cfg.batch_size
SEED = cfg.seed

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("DATA_ROOT       :", DATA_ROOT)
print("IMAGE_FILE      :", IMAGE_FILE)
print("MODEL_ROOT      :", MODEL_ROOT)
print("OUT_ROOT        :", OUT_ROOT)
print("PATCH_SIZE      :", PATCH_SIZE)
print("Patch FOV       :", PATCH_SIZE * PIXEL_SIZE_UM, "µm")
print("BEST_MODEL_PATH :", cfg.best_model_path)
print("FINAL_MODEL_PATH:", cfg.final_model_path)



In [ ]:
# ============================================================
# 3. Memory-mapped TIFF loading
# ============================================================

def load_memmap_tiff(path):
    """Load TIFF as a memory-mapped array. Do not use tifffile.imread for large hyperstacks."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Could not find TIFF file: {path}")

    arr = tiff.memmap(str(path))
    print("Loaded memory-mapped TIFF")
    print("  Path :", path)
    print("  Shape:", arr.shape)
    print("  Dtype:", arr.dtype)
    return arr

img_fov = load_memmap_tiff(IMAGE_FILE)
img = img_fov  # alias for convenience

if img_fov.ndim != 5:
    raise ValueError(f"Expected shape (T, Z, C, Y, X), got {img_fov.shape}")

print("IMAGE_FILE exists:", IMAGE_FILE.exists())
print("Image type:", type(img_fov))
print("Image shape:", img_fov.shape)
print("Image dtype:", img_fov.dtype)


In [ ]:
# ============================================================
# 4. Simple viewer: one timepoint, one Z plane, all channels
# ============================================================

def show_plane_all_channels(img, t=0, z=0, channel_names=None, percentile=(1, 99.8)):
    """Display all channels for a single time point and Z plane."""
    plane = img[t, z]  # (C, Y, X)
    C = plane.shape[0]

    if channel_names is None:
        channel_names = [f"Channel {i}" for i in range(C)]

    fig, axes = plt.subplots(1, C, figsize=(5 * C, 5))
    if C == 1:
        axes = [axes]

    for c in range(C):
        arr = np.asarray(plane[c])
        lo, hi = np.percentile(arr, percentile)
        axes[c].imshow(arr, cmap="gray", vmin=lo, vmax=hi)
        axes[c].set_title(f"{channel_names[c]} | t={t}, z={z}")
        axes[c].axis("off")

    plt.tight_layout()
    plt.show()

# Example:
show_plane_all_channels(
    img_fov,
    t=0,
    z=min(10, img_fov.shape[1] - 1),
    channel_names=["Membrane / context", "Nucleus", "NPC"],
)


In [ ]:
# ============================================================
# 5. Segmentation utilities: units, normalization, focus
# ============================================================

def area_um2_to_px(area_um2, pixel_size_um=PIXEL_SIZE_UM):
    """Convert area in µm² to pixels."""
    if area_um2 is None:
        return None
    return int(np.ceil(area_um2 / (pixel_size_um ** 2)))


def radius_um_to_px(radius_um, pixel_size_um=PIXEL_SIZE_UM):
    """Convert radius in µm to pixels."""
    if radius_um is None:
        return 0
    return max(1, int(np.ceil(radius_um / pixel_size_um)))


def normalize_01(img2d):
    """Normalize a 2D image to [0, 1] safely."""
    arr = img2d.astype("float32", copy=False)
    lo = np.nanmin(arr)
    hi = np.nanmax(arr)
    if hi <= lo:
        return np.zeros_like(arr, dtype="float32")
    return (arr - lo) / (hi - lo)


def normalize_channel(ch2d):
    """Normalize one channel for model input."""
    return normalize_01(ch2d).astype("float32")


def focus_score_variance_laplacian(img2d):
    """Variance of Laplacian focus score. Higher = sharper."""
    img_f = normalize_01(img2d)
    lap = ndi.laplace(img_f)
    return float(np.var(lap))


def z_plane_allowed(z, n_z, exclude_edge_z=cfg.edge_z_exclusion):
    """Exclude top/bottom Z planes where artifacts are common."""
    if exclude_edge_z is None or exclude_edge_z <= 0:
        return True
    return exclude_edge_z <= z < (n_z - exclude_edge_z)


def plane_passes_focus(nuc2d, min_focus_score=cfg.min_focus_score):
    """Return focus pass/fail and score."""
    score = focus_score_variance_laplacian(nuc2d)
    return score >= min_focus_score, score


def filter_objects_by_area(mask, pixel_size_um, min_area_um2=None, max_area_um2=None):
    """Keep connected objects within physical area limits."""
    labeled = measure.label(mask)
    clean = np.zeros_like(mask, dtype=bool)

    min_px = 0 if min_area_um2 is None else area_um2_to_px(min_area_um2, pixel_size_um)
    max_px = np.inf if max_area_um2 is None else area_um2_to_px(max_area_um2, pixel_size_um)

    for region in measure.regionprops(labeled):
        if min_px <= region.area <= max_px:
            clean[labeled == region.label] = True
    return clean

print("50 µm² nucleus minimum =", area_um2_to_px(50, PIXEL_SIZE_UM), "px")
print("512 px patch FOV =", PATCH_SIZE * PIXEL_SIZE_UM, "µm")


In [ ]:
# ============================================================
# 6. Data-driven NPC puncta clipping
# ============================================================

def estimate_npc_histogram_tail_cutoff(
    npc2d,
    bins=cfg.histogram_bins,
    smooth_window=cfg.histogram_smooth_window,
    tail_start_quantile=cfg.tail_start_quantile,
    min_clip_quantile=cfg.min_clip_quantile,
    max_clip_quantile=cfg.max_clip_quantile,
):
    """
    Estimate NPC puncta clipping cutoff from the bright histogram tail.

    This finds the steepest drop in the smoothed log-count histogram in the
    upper intensity tail, then bounds the resulting quantile.
    """
    npc_f = normalize_01(npc2d)
    vals = npc_f[np.isfinite(npc_f)].ravel()
    vals = vals[vals > 0]

    if vals.size == 0:
        debug = {
            "hist": None,
            "hist_smooth": None,
            "centers": None,
            "slope": None,
            "tail_start": None,
            "cutoff_value": 1.0,
            "cutoff_quantile": 1.0,
            "reason": "empty_nonzero_values",
        }
        return 1.0, 1.0, debug

    hist, edges = np.histogram(vals, bins=bins, range=(0, 1))
    centers = (edges[:-1] + edges[1:]) / 2

    smooth_window = int(max(3, smooth_window))
    if smooth_window % 2 == 0:
        smooth_window += 1

    kernel = np.ones(smooth_window, dtype="float32") / smooth_window
    hist_smooth = np.convolve(hist.astype("float32"), kernel, mode="same")
    log_hist = np.log1p(hist_smooth)
    slope = np.gradient(log_hist, centers)

    tail_start = np.quantile(vals, tail_start_quantile)
    tail_mask = centers >= tail_start

    if tail_mask.sum() < smooth_window:
        cutoff_quantile = max_clip_quantile
        cutoff_value = float(np.quantile(vals, cutoff_quantile))
        reason = "fallback_insufficient_tail_bins"
    else:
        tail_centers = centers[tail_mask]
        tail_slope = slope[tail_mask]
        slope_smooth = np.convolve(tail_slope, kernel, mode="same")

        tail_idx = int(np.argmin(slope_smooth))
        raw_cutoff_value = float(tail_centers[tail_idx])
        raw_cutoff_quantile = float(np.mean(vals <= raw_cutoff_value))

        cutoff_quantile = float(np.clip(raw_cutoff_quantile, min_clip_quantile, max_clip_quantile))
        cutoff_value = float(np.quantile(vals, cutoff_quantile))
        reason = "sliding_window_histogram_tail"

    debug = {
        "hist": hist,
        "hist_smooth": hist_smooth,
        "centers": centers,
        "slope": slope,
        "tail_start": float(tail_start),
        "cutoff_value": float(cutoff_value),
        "cutoff_quantile": float(cutoff_quantile),
        "reason": reason,
    }
    return cutoff_value, cutoff_quantile, debug


def clip_npc_puncta_data_driven(
    npc2d,
    bins=cfg.histogram_bins,
    smooth_window=cfg.histogram_smooth_window,
    tail_start_quantile=cfg.tail_start_quantile,
    min_clip_quantile=cfg.min_clip_quantile,
    max_clip_quantile=cfg.max_clip_quantile,
):
    """Clip bright NPC puncta using a data-driven histogram-tail estimate."""
    npc_f = normalize_01(npc2d)

    cutoff_value, cutoff_quantile, debug = estimate_npc_histogram_tail_cutoff(
        npc_f,
        bins=bins,
        smooth_window=smooth_window,
        tail_start_quantile=tail_start_quantile,
        min_clip_quantile=min_clip_quantile,
        max_clip_quantile=max_clip_quantile,
    )

    npc_clean = npc_f.copy()
    npc_clean[npc_clean > cutoff_value] = cutoff_value
    npc_clean = normalize_01(npc_clean)

    return npc_clean, cutoff_value, cutoff_quantile, debug


In [ ]:
# ============================================================
# 7. Clean segmentation functions
#    NPC puncta clipping + flat-field droplet foreground + watershed diagnostics
# ============================================================

def segment_droplet_from_npc(
    npc2d,
    pixel_size_um=PIXEL_SIZE_UM,
    use_data_driven_tail=cfg.use_data_driven_tail,
    puncta_clip_quantile=cfg.puncta_clip_quantile,
    histogram_bins=cfg.histogram_bins,
    histogram_smooth_window=cfg.histogram_smooth_window,
    tail_start_quantile=cfg.tail_start_quantile,
    min_clip_quantile=cfg.min_clip_quantile,
    max_clip_quantile=cfg.max_clip_quantile,
    blur_sigma=cfg.droplet_blur_sigma,
    illumination_sigma=cfg.droplet_illumination_sigma,
    foreground_threshold=cfg.droplet_foreground_threshold,
    min_droplet_area_um2=cfg.min_droplet_area_um2,
    hole_area_um2=cfg.droplet_hole_area_um2,
    closing_radius_um=cfg.droplet_closing_radius_um,
    watershed_min_distance_um=cfg.watershed_min_distance_um,
    return_debug=False,
    **unused_kwargs,
):
    """
    Segment droplet interiors from the NPC channel.

    Main steps:
        1. Normalize NPC image.
        2. Clip the bright NPC puncta tail using a data-driven histogram method.
        3. Low-pass blur to emphasize droplet-scale signal.
        4. Flat-field correct the low-pass image to remove broad FOV illumination bias.
        5. Threshold the flat-field image to generate a loose droplet foreground.
        6. Use distance transform + watershed for droplet diagnostics/separation.

    Notes:
        - `unused_kwargs` is intentional. It prevents old notebook calls from breaking
          if stale arguments such as local_block_size, local_offset, foreground_percentile,
          or erosion_radius_um are still passed from an older cell.
    """

    npc_f = normalize_01(npc2d)

    if use_data_driven_tail:
        npc_clean, clip_value, clip_quantile, puncta_debug = clip_npc_puncta_data_driven(
            npc2d,
            bins=histogram_bins,
            smooth_window=histogram_smooth_window,
            tail_start_quantile=tail_start_quantile,
            min_clip_quantile=min_clip_quantile,
            max_clip_quantile=max_clip_quantile,
        )
        clip_method = "data_driven_histogram_tail"
    else:
        clip_value = float(np.quantile(npc_f, puncta_clip_quantile))
        clip_quantile = float(puncta_clip_quantile)
        npc_clean = npc_f.copy()
        npc_clean[npc_clean > clip_value] = clip_value
        npc_clean = normalize_01(npc_clean)
        puncta_debug = {
            "cutoff_value": clip_value,
            "cutoff_quantile": clip_quantile,
            "reason": "fixed_quantile_fallback",
        }
        clip_method = "fixed_quantile"

    # Low-pass field: droplet-scale signal, not puncta signal.
    npc_blur = filters.gaussian(
        npc_clean,
        sigma=blur_sigma,
        preserve_range=True,
    )
    npc_field = normalize_01(npc_blur)

    # Flat-field correction: remove broad spatial illumination/intensity bias
    # before thresholding droplet foreground.
    illumination = filters.gaussian(
        npc_field,
        sigma=illumination_sigma,
        preserve_range=True,
    )
    illumination = illumination + 1e-6
    npc_flat = normalize_01(npc_field / illumination)

    # Loose foreground mask. This is intentionally simple because the flat-field
    # image is already normalized relative to local illumination.
    foreground = npc_flat > foreground_threshold

    if closing_radius_um and closing_radius_um > 0:
        foreground = morphology.binary_closing(
            foreground,
            morphology.disk(radius_um_to_px(closing_radius_um, pixel_size_um)),
        )

    if hole_area_um2 and hole_area_um2 > 0:
        foreground = morphology.remove_small_holes(
            foreground,
            area_threshold=area_um2_to_px(hole_area_um2, pixel_size_um),
        )

    foreground = morphology.remove_small_objects(
        foreground,
        min_size=area_um2_to_px(min_droplet_area_um2, pixel_size_um),
    )

    # Watershed diagnostics/separation. The class-1 droplet mask remains the full
    # foreground, while watershed_labels lets us inspect whether individual droplets
    # are being separated sensibly.
    distance = ndi.distance_transform_edt(foreground)
    min_distance_px = radius_um_to_px(watershed_min_distance_um, pixel_size_um)

    coords = peak_local_max(
        distance,
        min_distance=min_distance_px,
        labels=foreground,
        exclude_border=False,
    )

    markers = np.zeros(distance.shape, dtype=np.int32)
    for i, (y, x) in enumerate(coords, start=1):
        markers[y, x] = i

    if markers.max() > 0:
        watershed_labels = watershed(-distance, markers, mask=foreground)
    else:
        watershed_labels = measure.label(foreground)

    droplet_mask = foreground.astype(bool)

    if return_debug:
        return droplet_mask, {
            "npc_norm": npc_f,
            "npc_clean": npc_clean,
            "npc_blur": npc_blur,
            "npc_field": npc_field,
            "illumination": illumination,
            "npc_flat": npc_flat,
            "foreground": foreground,
            "distance": distance,
            "markers": markers,
            "watershed_labels": watershed_labels,
            "droplet_mask": droplet_mask,
            "clip_method": clip_method,
            "clip_value": float(clip_value),
            "clip_quantile": float(clip_quantile),
            "puncta_debug": puncta_debug,
            "threshold_method": "flatfield_absolute_threshold",
            "foreground_threshold": float(foreground_threshold),
            "illumination_sigma": float(illumination_sigma),
            "watershed_min_distance_px": int(min_distance_px),
            "n_markers": int(markers.max()),
            "min_droplet_area_px": area_um2_to_px(min_droplet_area_um2, pixel_size_um),
            "unused_kwargs": unused_kwargs,
        }

    return droplet_mask


def segment_nucleus_from_import(
    nuc2d,
    pixel_size_um=PIXEL_SIZE_UM,
    min_nucleus_area_um2=cfg.min_nucleus_area_um2,
    max_nucleus_area_um2=cfg.max_nucleus_area_um2,
    blur_sigma=cfg.nucleus_blur_sigma,
    threshold_method=cfg.nucleus_threshold_method,
    closing_radius_um=cfg.nucleus_closing_radius_um,
    hole_area_um2=cfg.nucleus_hole_area_um2,
    return_debug=False,
    **unused_kwargs,
):
    """Segment nuclei as filled solid objects for training labels."""
    nuc_f = normalize_01(nuc2d)
    nuc_blur = filters.gaussian(nuc_f, sigma=blur_sigma, preserve_range=True)

    if threshold_method == "otsu":
        thr = filters.threshold_otsu(nuc_blur)
    elif threshold_method == "yen":
        thr = filters.threshold_yen(nuc_blur)
    elif threshold_method == "li":
        thr = filters.threshold_li(nuc_blur)
    else:
        raise ValueError("threshold_method must be 'otsu', 'yen', or 'li'.")

    nucleus_mask = nuc_blur > thr

    if closing_radius_um and closing_radius_um > 0:
        nucleus_mask = morphology.binary_closing(
            nucleus_mask,
            morphology.disk(radius_um_to_px(closing_radius_um, pixel_size_um)),
        )

    nucleus_mask = ndi.binary_fill_holes(nucleus_mask)

    if hole_area_um2 and hole_area_um2 > 0:
        nucleus_mask = morphology.remove_small_holes(
            nucleus_mask,
            area_threshold=area_um2_to_px(hole_area_um2, pixel_size_um),
        )

    nucleus_mask = filter_objects_by_area(
        nucleus_mask,
        pixel_size_um=pixel_size_um,
        min_area_um2=min_nucleus_area_um2,
        max_area_um2=max_nucleus_area_um2,
    )

    nucleus_mask = nucleus_mask.astype(bool)

    if return_debug:
        return nucleus_mask, {
            "nuc_f": nuc_f,
            "nuc_blur": nuc_blur,
            "threshold": float(thr),
            "threshold_method": threshold_method,
            "min_nucleus_px": area_um2_to_px(min_nucleus_area_um2, pixel_size_um),
            "max_nucleus_px": area_um2_to_px(max_nucleus_area_um2, pixel_size_um),
            "unused_kwargs": unused_kwargs,
        }

    return nucleus_mask


def make_3class_label_full(droplet_mask, nucleus_mask, enforce_nucleus_inside_droplet=True):
    """
    Class labels:
        0 = image background
        1 = droplet interior
        2 = nucleus

    Nucleus overrides droplet.
    """
    if enforce_nucleus_inside_droplet:
        nucleus_mask = nucleus_mask & droplet_mask

    label = np.zeros(droplet_mask.shape, dtype=np.uint8)
    label[droplet_mask] = 1
    label[nucleus_mask] = 2
    return label


def build_training_label_plane(
    npc2d,
    nuc2d,
    pixel_size_um=PIXEL_SIZE_UM,
    min_focus_score=cfg.min_focus_score,
    use_focus_filter=cfg.use_focus_filter,
    droplet_kwargs=None,
    nucleus_kwargs=None,
    enforce_nucleus_inside_droplet=True,
):
    """Build one 3-class training label plane and return label, metadata, debug."""
    droplet_kwargs = droplet_kwargs or {}
    nucleus_kwargs = nucleus_kwargs or {}

    focus_pass, focus_score = plane_passes_focus(nuc2d, min_focus_score=min_focus_score)
    if use_focus_filter and not focus_pass:
        return None, {
            "focus_pass": False,
            "focus_score": focus_score,
            "droplet_area_px": 0,
            "nucleus_area_px": 0,
            "reason": "out_of_focus",
        }, None

    droplet_mask, droplet_debug = segment_droplet_from_npc(
        npc2d,
        pixel_size_um=pixel_size_um,
        return_debug=True,
        **droplet_kwargs,
    )

    nucleus_mask, nucleus_debug = segment_nucleus_from_import(
        nuc2d,
        pixel_size_um=pixel_size_um,
        return_debug=True,
        **nucleus_kwargs,
    )

    label = make_3class_label_full(
        droplet_mask,
        nucleus_mask,
        enforce_nucleus_inside_droplet=enforce_nucleus_inside_droplet,
    )

    final_nucleus_mask = label == 2

    metadata = {
        "focus_pass": True,
        "focus_score": focus_score,
        "droplet_area_px": int(droplet_mask.sum()),
        "nucleus_area_px": int(final_nucleus_mask.sum()),
        "npc_clip_method": droplet_debug.get("clip_method"),
        "npc_clip_value": droplet_debug.get("clip_value"),
        "npc_clip_quantile": droplet_debug.get("clip_quantile"),
        "foreground_threshold": droplet_debug.get("foreground_threshold"),
        "threshold_method": droplet_debug.get("threshold_method"),
        "n_markers": droplet_debug.get("n_markers"),
        "reason": "included",
    }

    debug = {
        **droplet_debug,
        **nucleus_debug,
        "droplet_mask": droplet_mask,
        "nucleus_mask_raw": nucleus_mask,
        "nucleus_mask": final_nucleus_mask,
        "label": label,
    }

    return label, metadata, debug


# Confirm that no legacy function signature is active.
print("Active segment_droplet_from_npc signature:")
print(inspect.signature(segment_droplet_from_npc))
print("Active segment_nucleus_from_import signature:")
print(inspect.signature(segment_nucleus_from_import))



In [ ]:
# ============================================================
# 8. QC: build and visualize one training plane
# ============================================================

# Choose a representative in-focus Z plane.
t = 9 if img_fov.shape[0] > 9 else img_fov.shape[0] - 1
z = min(10, img_fov.shape[1] - 1)

npc2d = img_fov[t, z, NPC_CH, :, :]
nuc2d = img_fov[t, z, NUC_CH, :, :]

label_test, metadata_test, debug_test = build_training_label_plane(
    npc2d=npc2d,
    nuc2d=nuc2d,
    pixel_size_um=PIXEL_SIZE_UM,
    droplet_kwargs={
        "use_data_driven_tail": cfg.use_data_driven_tail,
        "histogram_bins": cfg.histogram_bins,
        "histogram_smooth_window": cfg.histogram_smooth_window,
        "tail_start_quantile": cfg.tail_start_quantile,
        "min_clip_quantile": cfg.min_clip_quantile,
        "max_clip_quantile": cfg.max_clip_quantile,
        "blur_sigma": cfg.droplet_blur_sigma,
        "illumination_sigma": cfg.droplet_illumination_sigma,
        "foreground_threshold": cfg.droplet_foreground_threshold,
        "min_droplet_area_um2": cfg.min_droplet_area_um2,
        "hole_area_um2": cfg.droplet_hole_area_um2,
        "closing_radius_um": cfg.droplet_closing_radius_um,
        "watershed_min_distance_um": cfg.watershed_min_distance_um,
    },
    nucleus_kwargs={
        "min_nucleus_area_um2": cfg.min_nucleus_area_um2,
        "max_nucleus_area_um2": cfg.max_nucleus_area_um2,
        "blur_sigma": cfg.nucleus_blur_sigma,
        "threshold_method": cfg.nucleus_threshold_method,
        "closing_radius_um": cfg.nucleus_closing_radius_um,
        "hole_area_um2": cfg.nucleus_hole_area_um2,
    },
)

print(metadata_test)
if label_test is not None:
    print("Label classes/counts:", dict(zip(*np.unique(label_test, return_counts=True))))

    fig, ax = plt.subplots(2, 4, figsize=(20, 9))

    ax[0, 0].imshow(npc2d, cmap="gray")
    ax[0, 0].set_title("Raw NPC channel")

    ax[0, 1].imshow(debug_test["npc_blur"], cmap="gray")
    ax[0, 1].set_title("NPC clipped + low-pass blur")

    ax[0, 2].imshow(debug_test["npc_flat"], cmap="gray")
    ax[0, 2].set_title("Flat-field corrected NPC")

    ax[0, 3].imshow(debug_test["droplet_mask"], cmap="gray")
    ax[0, 3].set_title("Droplet training mask")

    ax[1, 0].imshow(nuc2d, cmap="gray")
    ax[1, 0].set_title("Raw nucleus channel")

    ax[1, 1].imshow(debug_test["nucleus_mask_raw"], cmap="gray")
    ax[1, 1].set_title("Filled nucleus mask before droplet constraint")

    ax[1, 2].imshow(debug_test["nucleus_mask"], cmap="gray")
    ax[1, 2].set_title("Final nucleus training mask")

    ax[1, 3].imshow(label_test, cmap="viridis", vmin=0, vmax=2)
    ax[1, 3].set_title("Final label: 0 bg, 1 droplet, 2 nucleus")

    for axis in ax.ravel():
        axis.axis("off")
    plt.tight_layout()
    plt.show()

    # Optional watershed diagnostic view
    plt.figure(figsize=(6, 5))
    plt.imshow(debug_test["watershed_labels"], cmap="nipy_spectral")
    plt.title("Watershed droplet labels diagnostic")
    plt.axis("off")
    plt.show()
else:
    print("Plane excluded:", metadata_test)



In [ ]:
# ============================================================
# 9. Patch extraction helpers
# ============================================================

def build_input_patch(mem_patch, nuc_patch, npc_patch):
    """
    Returns H×W×3 float32 in [0,1].

    Model input channel order:
        0 = nucleus channel
        1 = NPC channel
        2 = membrane/context channel
    """
    return np.stack(
        [
            normalize_channel(nuc_patch),
            normalize_channel(npc_patch),
            normalize_channel(mem_patch),
        ],
        axis=-1,
    ).astype("float32")


def clamp_patch_center(cy, cx, height, width, patch_size):
    """Clamp patch center so patch fits inside image."""
    half = patch_size // 2
    cy = int(np.clip(cy, half, height - half))
    cx = int(np.clip(cx, half, width - half))
    return cy, cx


def extract_2d_patch(arr2d, cy, cx, patch_size):
    """Extract a 2D patch centered at cy/cx."""
    half = patch_size // 2
    return arr2d[cy - half:cy + half, cx - half:cx + half]


def extract_plane_patch(img_fov, t, z, cy, cx, patch_size=cfg.patch_size):
    """Extract a normalized 3-channel input patch from memmap."""
    mem_patch = extract_2d_patch(img_fov[t, z, MEM_CH], cy, cx, patch_size)
    nuc_patch = extract_2d_patch(img_fov[t, z, NUC_CH], cy, cx, patch_size)
    npc_patch = extract_2d_patch(img_fov[t, z, NPC_CH], cy, cx, patch_size)
    return build_input_patch(mem_patch, nuc_patch, npc_patch)


def centers_from_mask(mask, max_centers=None, rng=None):
    """Return region centroids from connected components."""
    rng = rng or np.random.default_rng(SEED)
    labeled = measure.label(mask)
    centers = [(int(r.centroid[0]), int(r.centroid[1])) for r in measure.regionprops(labeled)]
    if max_centers is not None and len(centers) > max_centers:
        idx = rng.choice(len(centers), size=max_centers, replace=False)
        centers = [centers[i] for i in idx]
    return centers


def jitter_center(cy, cx, jitter_px, height, width, patch_size, rng):
    """Add random XY jitter and clamp to image bounds."""
    if jitter_px and jitter_px > 0:
        cy += int(rng.integers(-jitter_px, jitter_px + 1))
        cx += int(rng.integers(-jitter_px, jitter_px + 1))
    return clamp_patch_center(cy, cx, height, width, patch_size)


def random_centers_from_mask(mask, n, height, width, patch_size, rng):
    """Sample random centers from true pixels in a mask."""
    ys, xs = np.where(mask)
    if len(ys) == 0 or n <= 0:
        return []
    idx = rng.choice(len(ys), size=min(n, len(ys)), replace=False)
    centers = []
    for i in idx:
        centers.append(clamp_patch_center(int(ys[i]), int(xs[i]), height, width, patch_size))
    return centers


def generate_patch_centers(label, debug, patch_size=cfg.patch_size, patches_per_plane=cfg.patches_per_plane,
                           jitter_px=cfg.patch_jitter_px, rng=None):
    """
    Generate mixed patch centers:
        - mostly nucleus-anchored with jitter
        - some droplet-only/context
        - some hard-negative/background
    """
    rng = rng or np.random.default_rng(SEED)
    height, width = label.shape

    n_nucleus = int(round(patches_per_plane * cfg.nucleus_patch_fraction))
    n_droplet = int(round(patches_per_plane * cfg.droplet_patch_fraction))
    n_hard = max(0, patches_per_plane - n_nucleus - n_droplet)

    centers = []

    # Nucleus-anchored patches
    nuc_centers = centers_from_mask(debug["nucleus_mask"], rng=rng)
    if len(nuc_centers) > 0:
        take = min(n_nucleus, len(nuc_centers))
        idx = rng.choice(len(nuc_centers), size=take, replace=False)
        for i in idx:
            cy, cx = nuc_centers[i]
            centers.append(jitter_center(cy, cx, jitter_px, height, width, patch_size, rng))

    # Droplet/context patches not necessarily centered on nuclei
    droplet_only = debug["droplet_mask"] & ~debug["nucleus_mask"]
    centers.extend(random_centers_from_mask(droplet_only, n_droplet, height, width, patch_size, rng))

    # Hard-negative/background patches: from non-label pixels
    background = label == 0
    centers.extend(random_centers_from_mask(background, n_hard, height, width, patch_size, rng))

    # Fill any deficit with random valid positions
    half = patch_size // 2
    while len(centers) < patches_per_plane:
        cy = int(rng.integers(half, height - half))
        cx = int(rng.integers(half, width - half))
        centers.append((cy, cx))

    return centers[:patches_per_plane]


In [ ]:
# ============================================================
# 10. Build training patches from memory-mapped hyperstack
# ============================================================

def build_training_patches_from_hyperstack(
    img_fov,
    cfg=cfg,
    overwrite=False,
):
    """
    Build .npy image/label patches using the clean segmentation and mixed sampling strategy.

    This reads planes lazily from tifffile.memmap and saves:
        image patches: H×W×3 float32
        label patches: H×W uint8, classes 0/1/2
    """
    if overwrite:
        if cfg.image_patch_dir.exists():
            shutil.rmtree(cfg.image_patch_dir)
        if cfg.label_patch_dir.exists():
            shutil.rmtree(cfg.label_patch_dir)
        cfg.image_patch_dir.mkdir(parents=True, exist_ok=True)
        cfg.label_patch_dir.mkdir(parents=True, exist_ok=True)

    n_t, n_z, n_c, height, width = img_fov.shape
    rng = np.random.default_rng(cfg.seed)
    patch_id = 0
    plane_rows = []

    for t in tqdm(range(n_t), desc="Timepoints"):
        for z in range(n_z):
            if cfg.exclude_edge_z_planes and not z_plane_allowed(z, n_z, cfg.edge_z_exclusion):
                plane_rows.append({"t": t, "z": z, "included": False, "reason": "edge_z_excluded", "patches_saved": 0})
                continue

            npc2d = img_fov[t, z, cfg.npc_channel_idx]
            nuc2d = img_fov[t, z, cfg.nucleus_channel_idx]

            label, metadata, debug = build_training_label_plane(
                npc2d=npc2d,
                nuc2d=nuc2d,
                pixel_size_um=cfg.pixel_size_um,
                min_focus_score=cfg.min_focus_score,
                use_focus_filter=cfg.use_focus_filter,
                droplet_kwargs={
                    "use_data_driven_tail": cfg.use_data_driven_tail,
                    "histogram_bins": cfg.histogram_bins,
                    "histogram_smooth_window": cfg.histogram_smooth_window,
                    "tail_start_quantile": cfg.tail_start_quantile,
                    "min_clip_quantile": cfg.min_clip_quantile,
                    "max_clip_quantile": cfg.max_clip_quantile,
                    "blur_sigma": cfg.droplet_blur_sigma,
                    "illumination_sigma": cfg.droplet_illumination_sigma,
                    "foreground_threshold": cfg.droplet_foreground_threshold,
                    "min_droplet_area_um2": cfg.min_droplet_area_um2,
                    "hole_area_um2": cfg.droplet_hole_area_um2,
                    "closing_radius_um": cfg.droplet_closing_radius_um,
                    "watershed_min_distance_um": cfg.watershed_min_distance_um,
                },
                nucleus_kwargs={
                    "min_nucleus_area_um2": cfg.min_nucleus_area_um2,
                    "max_nucleus_area_um2": cfg.max_nucleus_area_um2,
                    "blur_sigma": cfg.nucleus_blur_sigma,
                    "threshold_method": cfg.nucleus_threshold_method,
                    "closing_radius_um": cfg.nucleus_closing_radius_um,
                    "hole_area_um2": cfg.nucleus_hole_area_um2,
                },
            )

            if label is None:
                plane_rows.append({
                    "t": t,
                    "z": z,
                    "included": False,
                    "reason": metadata.get("reason", "excluded"),
                    "focus_score": metadata.get("focus_score", np.nan),
                    "patches_saved": 0,
                })
                continue

            centers = generate_patch_centers(
                label,
                debug,
                patch_size=cfg.patch_size,
                patches_per_plane=cfg.patches_per_plane,
                jitter_px=cfg.patch_jitter_px,
                rng=rng,
            )

            patches_saved = 0
            for cy, cx in centers:
                y_patch = extract_2d_patch(label, cy, cx, cfg.patch_size)
                if y_patch.shape != (cfg.patch_size, cfg.patch_size):
                    continue

                # Skip patches with essentially no training signal.
                if np.mean(y_patch > 0) < cfg.min_label_fraction:
                    continue

                x_patch = extract_plane_patch(img_fov, t, z, cy, cx, cfg.patch_size)

                img_fname = cfg.image_patch_dir / f"img_t{t:03d}_z{z:03d}_y{cy:04d}_x{cx:04d}_p{patch_id:06d}.npy"
                lab_fname = cfg.label_patch_dir / f"lab_t{t:03d}_z{z:03d}_y{cy:04d}_x{cx:04d}_p{patch_id:06d}.npy"
                np.save(img_fname, x_patch.astype("float32"))
                np.save(lab_fname, y_patch.astype("uint8"))

                patch_id += 1
                patches_saved += 1

            plane_rows.append({
                "t": t,
                "z": z,
                "included": True,
                "reason": metadata.get("reason", "included"),
                "focus_score": metadata.get("focus_score", np.nan),
                "droplet_area_px": metadata.get("droplet_area_px", np.nan),
                "nucleus_area_px": metadata.get("nucleus_area_px", np.nan),
                "npc_clip_quantile": metadata.get("npc_clip_quantile", np.nan),
                "patches_saved": patches_saved,
            })

            del label, debug
            gc.collect()

    summary_df = pd.DataFrame(plane_rows)
    summary_path = cfg.training_root / "training_patch_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print("Saved patches:", patch_id)
    print("Summary:", summary_path)
    return summary_df

# Uncomment when ready to generate patches.
# summary_df = build_training_patches_from_hyperstack(img_fov, cfg=cfg, overwrite=False)



In [ ]:
# ============================================================
# 11. Patch QC helpers
# ============================================================

def list_patch_files(cfg=cfg):
    img_paths = sorted(cfg.image_patch_dir.glob("*.npy"))
    lab_paths = sorted(cfg.label_patch_dir.glob("*.npy"))
    print(f"Found {len(img_paths)} image patches and {len(lab_paths)} label patches.")
    return img_paths, lab_paths


def show_random_patch(cfg=cfg, seed=SEED):
    img_paths, lab_paths = list_patch_files(cfg)
    if not img_paths:
        print("No patches found. Run build_training_patches_from_hyperstack first.")
        return

    rng = np.random.default_rng(seed)
    i = int(rng.integers(0, len(img_paths)))
    x = np.load(img_paths[i])
    y = np.load(lab_paths[i])

    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    ax[0].imshow(x[..., 0], cmap="gray")
    ax[0].set_title("Input ch0: nucleus")
    ax[1].imshow(x[..., 1], cmap="gray")
    ax[1].set_title("Input ch1: NPC")
    ax[2].imshow(x[..., 2], cmap="gray")
    ax[2].set_title("Input ch2: membrane/context")
    ax[3].imshow(y, cmap="viridis", vmin=0, vmax=2)
    ax[3].set_title("Label")
    for a in ax:
        a.axis("off")
    plt.tight_layout()
    plt.show()
    print(img_paths[i].name)
    print("Class counts:", dict(zip(*np.unique(y, return_counts=True))))

# Example after patch generation:
# show_random_patch(cfg)


In [ ]:
# ============================================================
# 12. TensorFlow dataset from .npy patches
# ============================================================

def load_npy_pair_py(img_path, lab_path):
    if isinstance(img_path, np.ndarray):
        img_path = img_path.item()
    if isinstance(lab_path, np.ndarray):
        lab_path = lab_path.item()
    if isinstance(img_path, bytes):
        img_path = img_path.decode("utf-8")
    if isinstance(lab_path, bytes):
        lab_path = lab_path.decode("utf-8")

    img_arr = np.load(img_path).astype("float32")
    lab_arr = np.load(lab_path).astype("int32")
    return img_arr, lab_arr


def tf_load_npy_pair(img_path, lab_path):
    img_arr, lab_arr = tf.numpy_function(
        load_npy_pair_py,
        [img_path, lab_path],
        [tf.float32, tf.int32],
    )
    img_arr.set_shape((cfg.patch_size, cfg.patch_size, cfg.n_channels))
    lab_arr.set_shape((cfg.patch_size, cfg.patch_size))
    lab_oh = tf.one_hot(lab_arr, depth=cfg.num_classes, dtype=tf.float32)
    return img_arr, lab_oh


def make_dataset(img_paths, lab_paths, batch_size=cfg.batch_size, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((list(map(str, img_paths)), list(map(str, lab_paths))))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(img_paths), seed=cfg.seed, reshuffle_each_iteration=True)
    ds = ds.map(tf_load_npy_pair, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


def build_train_val_datasets(cfg=cfg):
    img_paths, lab_paths = list_patch_files(cfg)
    if len(img_paths) != len(lab_paths):
        raise ValueError(f"Image/label patch count mismatch: {len(img_paths)} vs {len(lab_paths)}")
    if len(img_paths) == 0:
        raise ValueError("No patches found. Generate patches before building datasets.")

    rng = np.random.default_rng(cfg.seed)
    idx = np.arange(len(img_paths))
    rng.shuffle(idx)

    n_val = max(1, int(round(len(idx) * cfg.validation_fraction)))
    val_idx = idx[:n_val]
    train_idx = idx[n_val:]

    train_img_paths = [img_paths[i] for i in train_idx]
    train_lab_paths = [lab_paths[i] for i in train_idx]
    val_img_paths = [img_paths[i] for i in val_idx]
    val_lab_paths = [lab_paths[i] for i in val_idx]

    train_ds = make_dataset(train_img_paths, train_lab_paths, cfg.batch_size, shuffle=True)
    val_ds = make_dataset(val_img_paths, val_lab_paths, cfg.batch_size, shuffle=False)

    print("Train patches:", len(train_img_paths))
    print("Val patches  :", len(val_img_paths))
    return train_ds, val_ds, train_img_paths, val_img_paths

# Uncomment after patches are generated:
# train_ds, val_ds, train_img_paths, val_img_paths = build_train_val_datasets(cfg)


In [ ]:
# ============================================================
# 13. U-Net model
# ============================================================

def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x


def encoder_block(x, filters):
    c = conv_block(x, filters)
    p = layers.MaxPooling2D((2, 2))(c)
    return c, p


def decoder_block(x, skip, filters):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x


def build_unet(input_shape=None, num_classes=None):
    if input_shape is None:
        input_shape = (cfg.patch_size, cfg.patch_size, cfg.n_channels)
    if num_classes is None:
        num_classes = cfg.num_classes

    inputs = layers.Input(shape=input_shape)

    c1, p1 = encoder_block(inputs, 32)
    c2, p2 = encoder_block(p1, 64)
    c3, p3 = encoder_block(p2, 128)
    c4, p4 = encoder_block(p3, 256)

    bn = conv_block(p4, 512)

    d1 = decoder_block(bn, c4, 256)
    d2 = decoder_block(d1, c3, 128)
    d3 = decoder_block(d2, c2, 64)
    d4 = decoder_block(d3, c1, 32)

    outputs = layers.Conv2D(num_classes, 1, activation="softmax")(d4)
    return models.Model(inputs, outputs, name=cfg.model_name)

model = build_unet()
model.summary()


In [ ]:
# ============================================================
# 14. Loss, metrics, callbacks, training
# ============================================================

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1, cfg.num_classes])
    y_pred_f = tf.reshape(y_pred, [-1, cfg.num_classes])
    intersection = tf.reduce_sum(y_true_f * y_pred_f, axis=0)
    denom = tf.reduce_sum(y_true_f + y_pred_f, axis=0)
    dice = (2.0 * intersection + smooth) / (denom + smooth)
    return tf.reduce_mean(dice)


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)


def combined_ce_dice_loss(y_true, y_pred):
    ce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
    ce = tf.reduce_mean(ce)
    return ce + dice_loss(y_true, y_pred)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=cfg.learning_rate),
    loss=combined_ce_dice_loss,
    metrics=[dice_coefficient, "categorical_accuracy"],
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(cfg.best_model_path),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_delta=1e-4,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1,
    ),
]

# Uncomment after train_ds and val_ds are created:
# history = model.fit(train_ds, validation_data=val_ds, epochs=cfg.epochs, callbacks=callbacks)
# model.save(cfg.final_model_path)


In [ ]:
# ============================================================
# 15. Visualize validation predictions
# ============================================================

def show_prediction_batch(model, dataset, n=2):
    for x_batch, y_batch in dataset.take(1):
        preds = model.predict(x_batch)
        pred_labels = np.argmax(preds, axis=-1)
        true_labels = np.argmax(y_batch.numpy(), axis=-1)

        n_show = min(n, x_batch.shape[0])
        for i in range(n_show):
            fig, ax = plt.subplots(1, 4, figsize=(16, 4))
            ax[0].imshow(x_batch[i, ..., 0], cmap="gray")
            ax[0].set_title("Input nucleus")
            ax[1].imshow(x_batch[i, ..., 1], cmap="gray")
            ax[1].set_title("Input NPC")
            ax[2].imshow(true_labels[i], cmap="viridis", vmin=0, vmax=2)
            ax[2].set_title("True label")
            ax[3].imshow(pred_labels[i], cmap="viridis", vmin=0, vmax=2)
            ax[3].set_title("Predicted label")
            for a in ax:
                a.axis("off")
            plt.tight_layout()
            plt.show()
        break

# Example after training:
# show_prediction_batch(model, val_ds, n=2)
